In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '\u9690\u85cf\u4ee3\u7801 \ud83d\udd12' : '\u663e\u793a\u4ee3\u7801 \ud83d\udd13';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">\u663e\u793a\u4ee3\u7801 \ud83d\udd13</button>
</div>
'''))


# 第10章 Section 10.3-10.4: 文件与抽象数据类型 (Files & Abstract Data Types)
## Cambridge International AS & A Level Computer Science (9618)

---

> **核心问题**: 程序运行时数据存在内存中, 程序关闭后数据就消失了. 如何持久化保存数据? 又如何用更高级的结构来组织和操作数据?

本节覆盖两大主题:
1. **文件操作 (File Handling)** --- 数据的持久化存储
2. **抽象数据类型 (ADTs)** --- Stack, Queue, Linked List

## Part 1: 文件操作 (File Handling)

### 为什么需要文件?

想象你玩了一个游戏, 打了很高的分数. 关闭游戏后, 如果分数没有 **保存到文件**, 下次打开就全没了!

| 存储位置 | 特点 | 比喻 |
|---------|------|------|
| **内存 (RAM)** | 快速, 但断电后丢失 (volatile) | 白板 --- 擦掉就没了 |
| **文件 (File)** | 较慢, 但持久保存 (non-volatile) | 笔记本 --- 写下来就在 |

### 文本文件 (Text File)
文本文件存储人类可读的字符序列, 每行以换行符 (line break) 分隔.

### 基本文件操作
| 操作 | Pseudocode | Python |
|------|-----------|--------|
| 打开文件 (读) | `OPENFILE "f.txt" FOR READ` | `open("f.txt", "r")` |
| 打开文件 (写) | `OPENFILE "f.txt" FOR WRITE` | `open("f.txt", "w")` |
| 打开文件 (追加) | `OPENFILE "f.txt" FOR APPEND` | `open("f.txt", "a")` |
| 读一行 | `READFILE "f.txt", line` | `f.readline()` |
| 写一行 | `WRITEFILE "f.txt", data` | `f.write(data)` |
| 关闭文件 | `CLOSEFILE "f.txt"` | `f.close()` |

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import os

# 文件写入
# 方法1: 传统方式
f = open("demo_file.txt", "w", encoding="utf-8")
f.write("Hello, this is line 1\n")
f.write("This is line 2\n")
f.write("Third line with number: 42\n")
f.close()
print("文件已写入 (传统方式)")

# 方法2: 使用 with 语句 (推荐! 自动关闭文件)
with open("demo_file.txt", "w", encoding="utf-8") as f:
    f.write("Hello, this is line 1\n")
    f.write("This is line 2\n")
    f.write("Third line with number: 42\n")
    f.write("Last line!\n")
print("文件已写入 (with 语句)")
print()

# 为什么 with 更好?
# 即使出错, with 也会自动关闭文件, 防止资源泄露
# 相当于 try...finally 的简洁写法


In [ ]:
# 文件读取
print("=== 方法1: 读取全部内容 ===")
with open("demo_file.txt", "r", encoding="utf-8") as f:
    content = f.read()
    print(repr(content))  # repr 显示换行符
print()

print("=== 方法2: 逐行读取 (最常用) ===")
with open("demo_file.txt", "r", encoding="utf-8") as f:
    line_num = 1
    for line in f:
        print(f"第{line_num}行: {line.strip()}")  # strip() 去掉换行符
        line_num += 1
print()

print("=== 方法3: 读取为列表 ===")
with open("demo_file.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
    print(f"共 {len(lines)} 行")
    for line in lines:
        print(f"  '{line.strip()}'")

In [ ]:
# 追加模式 (Append)
with open("demo_file.txt", "a", encoding="utf-8") as f:
    f.write("This line was appended!\n")

print("追加后的文件内容:")
with open("demo_file.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(f"  {line.strip()}")
print()

# 实际应用: 简单的成绩记录系统
print("=== 成绩记录系统 ===")

# 写入学生成绩
students_data = [
    "Alice,92,85,88",
    "Bob,78,90,72",
    "Carol,85,78,95",
]

with open("grades.csv", "w", encoding="utf-8") as f:
    f.write("Name,Math,English,Physics\n")  # 标题行
    for data in students_data:
        f.write(data + "\n")
print("成绩已保存到 grades.csv")

# 读取并处理
with open("grades.csv", "r", encoding="utf-8") as f:
    header = f.readline().strip()
    print(f"\n{header}")
    print("-" * 40)
    for line in f:
        parts = line.strip().split(",")
        name = parts[0]
        scores = [int(x) for x in parts[1:]]
        avg = sum(scores) / len(scores)
        print(f"{name}: 各科 {scores}, 平均 {avg:.1f}")

# 清理临时文件
os.remove("demo_file.txt")
os.remove("grades.csv")

## Part 2: 抽象数据类型 (Abstract Data Types - ADTs)

### 什么是 ADT?

**抽象数据类型 (Abstract Data Type)** 是一个数学模型, 定义了:
1. **数据** 的逻辑结构
2. 可以执行的 **操作** (operations)
3. 但 **不指定具体实现** (implementation)

### "黑箱" (Black Box) 概念

想想你每天用的自动售货机:
- 你知道 **WHAT**: 投币 -> 选择 -> 取货
- 你不知道 **HOW**: 内部的机械臂怎么工作

ADT 就像这样:
- 你知道 Stack 有 PUSH 和 POP 操作
- 你不需要知道内部用数组还是链表实现

### 为什么 ADT 重要?
| 优势 | 解释 |
|------|------|
| **抽象 (Abstraction)** | 隐藏复杂的实现细节 |
| **封装 (Encapsulation)** | 数据和操作绑定在一起 |
| **模块化 (Modularity)** | 可以替换实现而不影响使用者 |
| **可复用 (Reusability)** | 定义一次, 到处使用 |

## 3. 栈 (Stack) --- LIFO

### 什么是 Stack?

Stack 是一种 **LIFO (Last In, First Out)** 的数据结构: 最后放入的元素最先被取出.

### 现实类比: 一摞盘子

```
    +-------+
    | 盘子C |  <- TOP (最后放的, 最先取)
    +-------+
    | 盘子B |
    +-------+
    | 盘子A |  <- BOTTOM (最先放的, 最后取)
    +-------+
```

你只能:
- 在 **顶部** 放盘子 (PUSH)
- 从 **顶部** 取盘子 (POP)
- 看顶部是什么 (PEEK)

### Stack 的操作
| 操作 | 描述 | 时间复杂度 |
|------|------|-----------|
| `PUSH(item)` | 将元素压入栈顶 | O(1) |
| `POP()` | 移除并返回栈顶元素 | O(1) |
| `PEEK()` / `TOP()` | 查看栈顶元素 (不移除) | O(1) |
| `isEmpty()` | 检查栈是否为空 | O(1) |
| `isFull()` | 检查栈是否已满 (固定大小时) | O(1) |

### 真实世界的 Stack 应用
1. **撤销功能 (Undo)**: 每个操作 PUSH 进栈, Ctrl+Z 就是 POP
2. **括号匹配**: 编译器检查 `({[]})` 是否配对
3. **函数调用栈 (Call Stack)**: 程序执行函数时的嵌套调用
4. **浏览器后退**: 每访问一个页面 PUSH, 点后退就 POP

In [ ]:
# Stack 实现 (使用数组/列表)
class Stack:
    def __init__(self, max_size=10):
        self.stack = [None] * max_size  # 固定大小的数组
        self.top = -1                    # -1 表示空栈
        self.max_size = max_size

    def push(self, item):
        if self.is_full():
            print(f"Stack Overflow! 无法压入 {item}")
            return False
        self.top += 1
        self.stack[self.top] = item
        return True

    def pop(self):
        if self.is_empty():
            print("Stack Underflow! 栈为空")
            return None
        item = self.stack[self.top]
        self.stack[self.top] = None
        self.top -= 1
        return item

    def peek(self):
        if self.is_empty():
            print("栈为空, 无法 peek")
            return None
        return self.stack[self.top]

    def is_empty(self):
        return self.top == -1

    def is_full(self):
        return self.top == self.max_size - 1

    def size(self):
        return self.top + 1

    def display(self):
        if self.is_empty():
            print("[ 空栈 ]")
            return
        print("Stack (top -> bottom):")
        for i in range(self.top, -1, -1):
            marker = " <- TOP" if i == self.top else ""
            print(f"  | {self.stack[i]:>6} |{marker}")
        print("  +--------+")

# 演示
s = Stack(5)
print("=== Stack 操作演示 ===")
for val in [10, 20, 30, 40]:
    print(f"PUSH({val})")
    s.push(val)
s.display()
print()

print(f"PEEK: {s.peek()}")
print(f"POP: {s.pop()}")
print(f"POP: {s.pop()}")
print()
s.display()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Stack 可视化
def visualize_stack(stack_items, title="Stack", highlight_top=True):
    fig, ax = plt.subplots(figsize=(4, max(3, len(stack_items) * 0.8 + 1)))

    n = len(stack_items)
    if n == 0:
        ax.text(1, 1, 'Empty Stack', ha='center', va='center', fontsize=14, color='#95A5A6')
    else:
        for i in range(n):
            y = i * 0.7
            color = '#E74C3C' if (i == n - 1 and highlight_top) else '#3498DB'
            rect = patches.FancyBboxPatch((0.2, y), 1.6, 0.55,
                                           boxstyle="round,pad=0.05",
                                           facecolor=color, edgecolor='#2C3E50', linewidth=2)
            ax.add_patch(rect)
            ax.text(1, y + 0.275, str(stack_items[i]), ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white')
            if i == n - 1:
                ax.annotate('TOP', xy=(2.0, y + 0.275), fontsize=11,
                           fontweight='bold', color='#E74C3C',
                           arrowprops=dict(arrowstyle='->', color='#E74C3C'),
                           xytext=(2.5, y + 0.275))

    # 底部线
    ax.plot([0.2, 1.8], [-0.1, -0.1], color='#2C3E50', linewidth=3)

    ax.set_xlim(-0.2, 3.5)
    ax.set_ylim(-0.4, max(3, n * 0.7 + 0.5))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 展示 PUSH 和 POP 过程
print("1. 初始状态")
visualize_stack([])

print("2. PUSH 10, 20, 30")
visualize_stack([10, 20, 30])

print("3. POP -> 30 被移除")
visualize_stack([10, 20])

In [ ]:
# Stack 应用: 括号匹配检查器
def check_brackets(expression):
    stack = []
    matching = {')': '(', ']': '[', '}': '{'}
    opening = set('([{')
    closing = set(')]}')

    print(f"检查表达式: {expression}")
    for i, char in enumerate(expression):
        if char in opening:
            stack.append(char)
            print(f"  位置{i}: '{char}' -> PUSH  Stack: {stack}")
        elif char in closing:
            if not stack:
                print(f"  位置{i}: '{char}' -> 栈为空, 不匹配!")
                return False
            top = stack.pop()
            if top != matching[char]:
                print(f"  位置{i}: '{char}' -> POP '{top}', 不匹配!")
                return False
            print(f"  位置{i}: '{char}' -> POP '{top}', 匹配! Stack: {stack}")

    result = len(stack) == 0
    print(f"结果: {'匹配!' if result else '不匹配! 栈中还有: ' + str(stack)}")
    return result

print("=== 测试1: 正确的括号 ===")
check_brackets("{[()]}")
print()
print("=== 测试2: 错误的括号 ===")
check_brackets("{[(])}")
print()
print("=== 测试3: 缺少右括号 ===")
check_brackets("(()")

## 4. 队列 (Queue) --- FIFO

### 什么是 Queue?

Queue 是一种 **FIFO (First In, First Out)** 的数据结构: 最先放入的元素最先被取出.

### 现实类比: 排队买票

```
出口 <-- [人A] [人B] [人C] [人D] <-- 入口
         FRONT                  REAR
```

- 新来的人从 **后面 (rear)** 排队 --- ENQUEUE
- 最前面的人先离开 --- DEQUEUE

### Queue 的操作
| 操作 | 描述 | 时间复杂度 |
|------|------|-----------|
| `ENQUEUE(item)` | 将元素加入队尾 | O(1) |
| `DEQUEUE()` | 移除并返回队首元素 | O(1) |
| `PEEK()` / `FRONT()` | 查看队首元素 | O(1) |
| `isEmpty()` | 检查队列是否为空 | O(1) |

### 循环队列 (Circular Queue) 的概念

用普通数组实现队列有个问题: DEQUEUE 后前面的空间浪费了!

**解决方案**: 循环队列 --- 把数组首尾相连, 像一个圆环:
```
当 rear 到达末尾时, 绕回到开头!
rear = (rear + 1) % max_size
```

### Queue 的真实应用
1. **打印队列**: 多个文档按顺序打印
2. **任务调度**: 操作系统分配 CPU 时间
3. **消息队列**: 网络通信中的消息缓冲
4. **BFS 算法**: 广度优先搜索使用队列

In [ ]:
# Queue 实现 (循环队列)
class Queue:
    def __init__(self, max_size=10):
        self.queue = [None] * max_size
        self.front = 0
        self.rear = -1
        self.count = 0
        self.max_size = max_size

    def enqueue(self, item):
        if self.is_full():
            print(f"Queue Full! 无法加入 {item}")
            return False
        self.rear = (self.rear + 1) % self.max_size
        self.queue[self.rear] = item
        self.count += 1
        return True

    def dequeue(self):
        if self.is_empty():
            print("Queue Empty! 无法取出")
            return None
        item = self.queue[self.front]
        self.queue[self.front] = None
        self.front = (self.front + 1) % self.max_size
        self.count -= 1
        return item

    def peek(self):
        if self.is_empty():
            return None
        return self.queue[self.front]

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.max_size

    def display(self):
        if self.is_empty():
            print("[ 空队列 ]")
            return
        print("Queue (front -> rear):", end=" ")
        idx = self.front
        for i in range(self.count):
            marker = ""
            if i == 0:
                marker = "(F)"
            if i == self.count - 1:
                marker += "(R)"
            print(f"[{self.queue[idx]}{marker}]", end=" ")
            idx = (idx + 1) % self.max_size
        print()

# 演示
q = Queue(5)
print("=== Queue 操作演示 ===")
for val in ["Task_A", "Task_B", "Task_C"]:
    print(f"ENQUEUE('{val}')")
    q.enqueue(val)
q.display()
print()

print(f"DEQUEUE: {q.dequeue()}")
print(f"DEQUEUE: {q.dequeue()}")
q.display()
print()

# 循环特性演示
print("=== 循环特性 ===")
q.enqueue("Task_D")
q.enqueue("Task_E")
q.enqueue("Task_F")  # 绕回到数组开头!
q.display()
print(f"内部数组状态: {q.queue}")
print("注意: Task_F 绕回到了数组的前面位置!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Queue 可视化
def visualize_queue(items, front_label=None, rear_label=None):
    fig, ax = plt.subplots(figsize=(max(6, len(items) * 2), 3))

    n = len(items)
    for i in range(n):
        color = '#2ECC71' if i == 0 else ('#E74C3C' if i == n - 1 else '#3498DB')
        rect = patches.FancyBboxPatch((i * 2, 0.5), 1.6, 1.2,
                                       boxstyle="round,pad=0.05",
                                       facecolor=color, edgecolor='#2C3E50', linewidth=2)
        ax.add_patch(rect)
        ax.text(i * 2 + 0.8, 1.1, str(items[i]), ha='center', va='center',
                fontsize=12, fontweight='bold', color='white')

    if n > 0:
        ax.text(0.8, 2.0, 'FRONT', ha='center', fontsize=11, fontweight='bold', color='#2ECC71')
        ax.text((n - 1) * 2 + 0.8, 2.0, 'REAR', ha='center', fontsize=11, fontweight='bold', color='#E74C3C')

        # 箭头: DEQUEUE 方向
        ax.annotate('', xy=(-0.5, 1.1), xytext=(0.1, 1.1),
                    arrowprops=dict(arrowstyle='->', color='#2ECC71', lw=2))
        ax.text(-0.8, 1.1, 'OUT', ha='center', va='center', fontsize=10, color='#2ECC71', fontweight='bold')

        # 箭头: ENQUEUE 方向
        ax.annotate('', xy=((n - 1) * 2 + 1.7, 1.1), xytext=((n - 1) * 2 + 2.3, 1.1),
                    arrowprops=dict(arrowstyle='<-', color='#E74C3C', lw=2))
        ax.text((n - 1) * 2 + 2.6, 1.1, 'IN', ha='center', va='center', fontsize=10, color='#E74C3C', fontweight='bold')

    ax.set_xlim(-1.5, max(6, n * 2 + 1.5))
    ax.set_ylim(-0.2, 2.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Queue: FIFO (First In, First Out)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_queue(["A", "B", "C", "D"])

## 5. 链表 (Linked List)

### 为什么需要链表?

数组有一个关键限制: **大小固定, 插入/删除慢**.

| 操作 | 数组 | 链表 |
|------|------|------|
| 随机访问 | O(1) 很快 | O(n) 需遍历 |
| 在头部插入 | O(n) 需移动所有元素 | O(1) 只改指针 |
| 在中间插入 | O(n) 需移动后续元素 | O(1) 找到位置后 |
| 删除 | O(n) 需移动后续元素 | O(1) 找到位置后 |
| 内存 | 连续分配 | 不需要连续 |

### 链表的结构

链表由 **节点 (Node)** 组成, 每个节点包含:
1. **数据 (Data)**: 存储的值
2. **指针 (Pointer / Next)**: 指向下一个节点的地址

```
HEAD
  |
  v
+------+------+    +------+------+    +------+------+
| Data | Next | -> | Data | Next | -> | Data | Next | -> NULL
|  10  |   -------> |  20  |   -------> |  30  | NULL |
+------+------+    +------+------+    +------+------+
```

- **HEAD**: 指向第一个节点的指针
- **NULL**: 表示链表结束 (最后一个节点的 Next)

### 用数组实现链表 (A Level 考试方式)

在 A Level 中, 链表常用 **两个平行数组** 实现:
- `data[]`: 存储节点的数据
- `pointer[]`: 存储下一个节点的索引 (-1 表示结束)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 链表可视化
def visualize_linked_list(data_list, title="Linked List"):
    fig, ax = plt.subplots(figsize=(max(8, len(data_list) * 3.5), 3))

    n = len(data_list)
    for i in range(n):
        x = i * 3
        # 数据部分
        rect_data = patches.FancyBboxPatch((x, 0.4), 1.2, 1.2,
                                            boxstyle="round,pad=0.05",
                                            facecolor='#3498DB', edgecolor='#2C3E50', linewidth=2)
        ax.add_patch(rect_data)
        ax.text(x + 0.6, 1.0, str(data_list[i]), ha='center', va='center',
                fontsize=14, fontweight='bold', color='white')
        ax.text(x + 0.6, 0.5, 'Data', ha='center', va='center', fontsize=8, color='#BDC3C7')

        # 指针部分
        rect_ptr = patches.FancyBboxPatch((x + 1.2, 0.4), 0.8, 1.2,
                                           boxstyle="round,pad=0.05",
                                           facecolor='#E67E22', edgecolor='#2C3E50', linewidth=2)
        ax.add_patch(rect_ptr)

        if i < n - 1:
            # 箭头指向下一个节点
            ax.annotate('', xy=((i + 1) * 3, 1.0), xytext=(x + 2.0, 1.0),
                        arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))
        else:
            ax.text(x + 1.6, 1.0, 'NULL', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')

    # HEAD 标签
    if n > 0:
        ax.annotate('HEAD', xy=(0, 1.0), xytext=(-1.5, 1.8),
                    fontsize=12, fontweight='bold', color='#E74C3C',
                    arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

    ax.set_xlim(-2, max(8, n * 3 + 1))
    ax.set_ylim(-0.3, 2.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_linked_list([10, 20, 30, 40, 50])

In [ ]:
# 用数组实现链表 (A Level 方式)
class LinkedListArray:
    def __init__(self, max_size=10):
        self.data = [None] * max_size
        self.pointer = [-1] * max_size  # -1 = null
        self.head = -1          # -1 = empty list
        self.free = 0           # 空闲链表的头
        self.max_size = max_size
        # 初始化空闲链表
        for i in range(max_size - 1):
            self.pointer[i] = i + 1
        self.pointer[max_size - 1] = -1

    def _get_free(self):
        if self.free == -1:
            return -1
        idx = self.free
        self.free = self.pointer[self.free]
        return idx

    def _return_free(self, idx):
        self.pointer[idx] = self.free
        self.data[idx] = None
        self.free = idx

    def insert_at_head(self, value):
        new_idx = self._get_free()
        if new_idx == -1:
            print("链表已满!")
            return
        self.data[new_idx] = value
        self.pointer[new_idx] = self.head
        self.head = new_idx
        print(f"插入 {value} 到头部 (存储在索引 {new_idx})")

    def delete(self, value):
        if self.head == -1:
            print("链表为空!")
            return False
        # 删除头节点
        if self.data[self.head] == value:
            old_head = self.head
            self.head = self.pointer[self.head]
            self._return_free(old_head)
            print(f"删除 {value} (原头节点)")
            return True
        # 搜索
        prev = self.head
        curr = self.pointer[self.head]
        while curr != -1:
            if self.data[curr] == value:
                self.pointer[prev] = self.pointer[curr]
                self._return_free(curr)
                print(f"删除 {value}")
                return True
            prev = curr
            curr = self.pointer[curr]
        print(f"{value} 不在链表中")
        return False

    def traverse(self):
        if self.head == -1:
            print("链表为空")
            return
        result = []
        curr = self.head
        while curr != -1:
            result.append(f"{self.data[curr]}(idx:{curr})")
            curr = self.pointer[curr]
        print("链表: " + " -> ".join(result) + " -> NULL")

    def display_arrays(self):
        print(f"{'Index':<8}{'Data':<10}{'Pointer':<10}")
        print("-" * 28)
        for i in range(self.max_size):
            d = str(self.data[i]) if self.data[i] is not None else '-'
            print(f"{i:<8}{d:<10}{self.pointer[i]:<10}")
        print(f"HEAD = {self.head}, FREE = {self.free}")

# 演示
ll = LinkedListArray(6)
print("=== 链表操作 (数组实现) ===")
ll.insert_at_head(30)
ll.insert_at_head(20)
ll.insert_at_head(10)
ll.insert_at_head(5)
print()

ll.traverse()
print()

print("=== 内部数组状态 ===")
ll.display_arrays()
print()

ll.delete(20)
print()
ll.traverse()

In [ ]:
# 链表操作可视化: 插入节点
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

def draw_ll(ax, nodes, title, highlight_idx=-1, new_node=None, new_pos=-1):
    n = len(nodes)
    for i in range(n):
        x = i * 3
        color = '#2ECC71' if i == highlight_idx else '#3498DB'
        rect = patches.FancyBboxPatch((x, 0.2), 1.3, 1.0,
                                       boxstyle="round,pad=0.05",
                                       facecolor=color, edgecolor='#2C3E50', linewidth=2)
        ax.add_patch(rect)
        ax.text(x + 0.65, 0.7, str(nodes[i]), ha='center', va='center',
                fontsize=13, fontweight='bold', color='white')
        if i < n - 1:
            ax.annotate('', xy=((i + 1) * 3, 0.7), xytext=(x + 1.3, 0.7),
                        arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2))

    if new_node is not None:
        x = new_pos * 3
        rect = patches.FancyBboxPatch((x, 1.8), 1.3, 1.0,
                                       boxstyle="round,pad=0.05",
                                       facecolor='#E74C3C', edgecolor='#2C3E50', linewidth=2)
        ax.add_patch(rect)
        ax.text(x + 0.65, 2.3, str(new_node), ha='center', va='center',
                fontsize=13, fontweight='bold', color='white')
        ax.text(x + 0.65, 3.0, 'NEW', ha='center', fontsize=10,
                fontweight='bold', color='#E74C3C')

    ax.set_xlim(-1, max(8, n * 3 + 2))
    ax.set_ylim(-0.3, 3.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', loc='left')

draw_ll(axes[0], [10, 30, 40], "Step 1: Original list - 要在 10 和 30 之间插入 20")
draw_ll(axes[1], [10, 30, 40], "Step 2: 创建新节点 20, 准备插入", highlight_idx=0, new_node=20, new_pos=1)
draw_ll(axes[2], [10, 20, 30, 40], "Step 3: 调整指针, 插入完成!", highlight_idx=1)

plt.suptitle('Linked List: Insert Operation (链表插入操作)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 数据结构对比 (Comparison)

| 特性 | Array (数组) | Stack (栈) | Queue (队列) | Linked List (链表) |
|------|-------------|-----------|-------------|-------------------|
| **存取方式** | 随机访问 | LIFO | FIFO | 顺序访问 |
| **访问时间** | O(1) | O(1) 仅栈顶 | O(1) 仅首尾 | O(n) |
| **插入时间** | O(n) 中间 | O(1) 顶部 | O(1) 尾部 | O(1) 已知位置 |
| **删除时间** | O(n) 中间 | O(1) 顶部 | O(1) 头部 | O(1) 已知位置 |
| **大小** | 固定 | 固定 (可动态) | 固定 (可动态) | 动态 |
| **内存** | 连续 | 连续 | 连续 | 不需连续 |
| **典型用途** | 存储列表 | 撤销, 递归 | 任务排队 | 动态插删 |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 可视化: 各数据结构的操作时间对比
structures = ['Array', 'Stack', 'Queue', 'Linked List']
operations = ['Access', 'Insert (head)', 'Insert (mid)', 'Delete (head)', 'Search']

# 时间复杂度 (1=O(1), 2=O(n), 0.5=amortized)
data = np.array([
    [1, 2, 2, 2, 2],  # Array
    [1, 1, 0, 1, 0],  # Stack (只支持部分操作)
    [1, 0, 0, 1, 0],  # Queue (只支持部分操作)
    [2, 1, 1, 1, 2],  # Linked List
])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(operations))
width = 0.18
colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']

for i in range(len(structures)):
    bars = ax.bar(x + i * width, data[i], width, label=structures[i],
                  color=colors[i], edgecolor='white', linewidth=1)

ax.set_xlabel('Operation', fontsize=12)
ax.set_ylabel('Time Complexity (1=O(1), 2=O(n), 0=N/A)', fontsize=11)
ax.set_title('Data Structure Operation Comparison (数据结构操作对比)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(operations, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 2.8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 综合应用: 简易任务管理器

结合 Stack (撤销功能) 和 Queue (任务队列) 的实际应用.

In [ ]:
# 综合应用: 任务管理器
class TaskManager:
    def __init__(self):
        self.task_queue = []      # Queue: 待处理任务
        self.undo_stack = []      # Stack: 已完成任务 (可撤销)
        self.completed = []       # 已完成列表

    def add_task(self, task):
        self.task_queue.append(task)  # ENQUEUE
        print(f"[+] 新任务加入队列: '{task}'")

    def process_next(self):
        if not self.task_queue:
            print("[!] 任务队列为空!")
            return
        task = self.task_queue.pop(0)  # DEQUEUE
        self.undo_stack.append(task)   # PUSH to undo stack
        self.completed.append(task)
        print(f"[v] 完成任务: '{task}'")

    def undo_last(self):
        if not self.undo_stack:
            print("[!] 没有可撤销的任务!")
            return
        task = self.undo_stack.pop()   # POP from undo stack
        self.completed.remove(task)
        self.task_queue.insert(0, task)  # 放回队列头部
        print(f"[<] 撤销任务: '{task}' (放回队列)")

    def status(self):
        print("\n" + "=" * 45)
        print(f"  待处理 (Queue): {self.task_queue}")
        print(f"  已完成: {self.completed}")
        print(f"  可撤销 (Stack): {[t for t in self.undo_stack]}")
        print("=" * 45)

# 演示
tm = TaskManager()
print("=== 任务管理器演示 ===")
tm.add_task("写数学作业")
tm.add_task("复习英语单词")
tm.add_task("完成CS项目")
tm.add_task("准备物理实验")
tm.status()

print()
tm.process_next()  # 完成 "写数学作业"
tm.process_next()  # 完成 "复习英语单词"
tm.status()

print()
tm.undo_last()     # 撤销 "复习英语单词"
tm.status()

## 8. 练习题 (Practice Exercises)

### 练习 1: 文件操作
编写程序:
1. 创建一个文件, 写入 5 个学生的姓名和分数
2. 读取文件, 计算平均分
3. 将结果追加到文件末尾

### 练习 2: Stack
用 Stack 实现一个 **十进制转二进制** 的算法:
- 将十进制数不断除以 2
- 每次的余数 PUSH 进栈
- 最后依次 POP 出来就是二进制表示

### 练习 3: Queue
模拟一个打印队列:
- 有 5 个打印任务, 每个有名称和页数
- 按 FIFO 顺序处理
- 每处理一个, 显示剩余任务

### 练习 4: Linked List
在链表中实现 **搜索** 功能: 给定一个值, 返回它在链表中的位置 (第几个节点).

### 练习 5: 综合题
Stack 和 Queue 的区别是什么? 各举两个真实世界的应用场景. 解释为什么这些场景适合使用对应的数据结构.

In [ ]:
# 练习参考答案

# 练习 2: 十进制转二进制 (用 Stack)
def decimal_to_binary(n):
    if n == 0:
        return "0"
    stack = []
    original = n
    print(f"将 {n} 转为二进制:")
    while n > 0:
        remainder = n % 2
        stack.append(remainder)
        print(f"  {n} / 2 = {n // 2} 余 {remainder} -> PUSH({remainder})")
        n = n // 2

    # POP 出所有元素
    binary = ""
    print("\nPOP 出栈:")
    while stack:
        bit = stack.pop()
        binary += str(bit)
        print(f"  POP -> {bit}")

    print(f"\n结果: {original} (十进制) = {binary} (二进制)")
    # 验证
    print(f"验证: int('{binary}', 2) = {int(binary, 2)}")
    return binary

decimal_to_binary(42)
print()

# 练习 3: 打印队列
print("=== 练习 3: 打印队列模拟 ===")
print_queue = [
    ("Report.pdf", 5),
    ("Essay.docx", 12),
    ("Photo.jpg", 1),
    ("Slides.pptx", 8),
    ("Code.py", 3),
]
print(f"初始队列: {[t[0] for t in print_queue]}")
while print_queue:
    task_name, pages = print_queue.pop(0)  # DEQUEUE
    print(f"  正在打印: {task_name} ({pages} 页)")
    remaining = [t[0] for t in print_queue]
    print(f"  剩余: {remaining if remaining else '(无)'}")
print("所有任务完成!")

---
## 本节要点回顾

### 文件操作
1. 文件用于 **持久化存储** 数据
2. 三种模式: READ, WRITE, APPEND
3. Python 推荐使用 `with` 语句自动管理文件

### 抽象数据类型 (ADTs)
4. ADT = 数据 + 操作, 隐藏实现细节 ("黑箱")
5. **Stack (LIFO)**: PUSH, POP, PEEK --- 撤销, 括号匹配, 调用栈
6. **Queue (FIFO)**: ENQUEUE, DEQUEUE --- 打印队列, 任务调度
7. **Linked List**: 节点 + 指针, 动态大小, 高效插删
8. 选择数据结构取决于你需要什么操作效率最高

> **恭喜!** 你已经完成了第 10 章的所有内容. 这些数据结构是计算机科学的基石, 你将在后续章节中不断用到它们!